In [1]:
from google.colab import drive
drive.mount('/content/drive')

import sys
PROJECT_ROOT = '/content/drive/MyDrive/HouseLLM'
sys.path.insert(0, f'{PROJECT_ROOT}/labeler')
sys.path.insert(0, f'{PROJECT_ROOT}/inference')
sys.path.insert(0, f'{PROJECT_ROOT}/evaluation')

print("Paths set up.")

Mounted at /content/drive
Paths set up.


In [2]:
!pip install -q transformers torch pydantic openai
!pip install -q --upgrade lm-format-enforcer


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.4/45.4 kB 4.0 MB/s eta 0:00:00


In [3]:
import lmformatenforcer, os

file_path = os.path.join(
    os.path.dirname(lmformatenforcer.__file__),
    'integrations',
    'transformers.py'
)
print(f"Patching: {file_path}")

with open(file_path) as f:
    content = f.read()

# The bad import that fails:
old_import = 'from transformers.tokenization_utils import PreTrainedTokenizerBase'
# The version that works on every transformers release:
new_import = 'from transformers import PreTrainedTokenizerBase'

if old_import in content:
    content = content.replace(old_import, new_import)
    with open(file_path, 'w') as f:
        f.write(content)
    print("Patched successfully.")
else:
    print("Bad import not found — file may already be patched or have different content.")
    print("First 30 lines for inspection:")
    for i, line in enumerate(content.split('\n')[:30], 1):
        print(f"  {i:3d}: {line}")

Patching: /usr/local/lib/python3.12/dist-packages/lmformatenforcer/integrations/transformers.py
Patched successfully.


In [4]:
from google.colab import userdata
import os

os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')
os.environ['OPENAI_API_KEY'] = userdata.get('OPENAI_API_KEY')

# Log in to Hugging Face for model download
from huggingface_hub import login
login(token=os.environ['HF_TOKEN'])
print("Authenticated.")

Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


Authenticated.


In [5]:
import torch
print(f"CUDA available: {torch.cuda.is_available()}")
print(f"Device: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'}")
print(f"Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

CUDA available: True
Device: NVIDIA L4
Memory: 23.7 GB


In [6]:
from pathlib import Path

schema_path = Path(f'{PROJECT_ROOT}/inference/schema.py')
labels_path = Path(f'{PROJECT_ROOT}/labeler/labels_clean.jsonl')
fs_path     = Path(f'{PROJECT_ROOT}/labeler/few_shot_examples.py')
icd_path    = Path(f'{PROJECT_ROOT}/inference/icd10_filtered.csv')

print(f"schema.py:             {'OK' if schema_path.exists() else 'MISSING'}")
print(f"labels_clean.jsonl:    {'OK' if labels_path.exists() else 'MISSING'}")
print(f"few_shot_examples.py:  {'OK' if fs_path.exists() else 'MISSING (smoke test will skip few-shot)'}")
print(f"icd10_filtered.csv:    {'OK' if icd_path.exists() else 'MISSING (hard condition will fall back to medium)'}")

assert schema_path.exists(), "Cannot proceed without schema.py"
assert labels_path.exists(), "Cannot proceed without labels_clean.jsonl"
print("\nPre-flight passed.")

schema.py:             OK
labels_clean.jsonl:    OK
few_shot_examples.py:  OK
icd10_filtered.csv:    OK

Pre-flight passed.


In [ ]:
# import subprocess
# result = subprocess.run(
#     ['find', PROJECT_ROOT, '-name', 'schema.py', '-not', '-path', '*/.*'],
#     capture_output=True, text=True
# )
# print("Found schema.py at:")
# print(result.stdout if result.stdout else "(nothing found anywhere under PROJECT_ROOT)")

In [7]:
import sys, json
sys.path.insert(0, f'{PROJECT_ROOT}/labeler')

try:
    from few_shot_examples import FEW_SHOT_IDS, FEW_SHOT_EXAMPLES
    print(f"Few-shot: {len(FEW_SHOT_EXAMPLES)} examples loaded, "
          f"excluding {sorted(FEW_SHOT_IDS)} from smoke test.")
except ImportError:
    FEW_SHOT_IDS = set()
    FEW_SHOT_EXAMPLES = None
    print("No few-shot file — smoke test will run without few-shot prompting.")

records = []
with open(labels_path) as f:
    for line in f:
        r = json.loads(line)
        if r['encounter_id'] not in FEW_SHOT_IDS:
            records.append(r)

sample = records[:3]
print(f"\nSmoke testing on: {[r['encounter_id'] for r in sample]}")

Few-shot: 2 examples loaded, excluding ['D2N001', 'D2N003'] from smoke test.

Smoke testing on: ['D2N002', 'D2N004', 'D2N005']


In [8]:
from llama_runner import LlamaRunner
import time

runner = LlamaRunner(condition="baseline")

for r in sample:
    t0 = time.time()
    pred = runner.generate(r['dialogue'])
    dt = time.time() - t0
    print(f"\n=== {r['encounter_id']}  ({dt:.1f}s) ===")
    print("--- GOLD ---")
    print(json.dumps(r['label'], indent=2))
    print("--- PRED ---")
    print(json.dumps(pred, indent=2))

Loading meta-llama/Llama-3.2-3B-Instruct for condition=baseline...


`torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/254 [00:00<?, ?it/s]

The following generation flags are not valid and may be ignored: ['temperature', 'top_p']. Set `TRANSFORMERS_VERBOSITY=info` for more details.



=== D2N002  (12.5s) ===
--- GOLD ---
{
  "name": "Andrew",
  "age": 62,
  "symptoms": [
    "joint pain",
    "knee pain"
  ],
  "duration": "over the weekend",
  "negated_symptoms": [
    "fever",
    "chills",
    "muscle aches",
    "nausea",
    "vomiting",
    "diarrhea",
    "headache",
    "fatigue",
    "weight gain"
  ],
  "history": [
    "kidney transplant",
    "hypothyroidism",
    "arthritis",
    "polycystic kidneys"
  ],
  "diagnosis": [
    "acute exacerbation of arthritis",
    "hypothyroidism"
  ],
  "treatment": [
    {
      "type": "medication",
      "detail": "ultram 50 mg every 6 hours as needed"
    },
    {
      "type": "medication",
      "detail": "continue synthroid"
    },
    {
      "type": "test",
      "detail": "order autoimmune panel"
    },
    {
      "type": "test",
      "detail": "order thyroid panel"
    },
    {
      "type": "counseling",
      "detail": "take it easy for now"
    },
    {
      "type": "follow_up",
      "detail": "talk a

In [9]:
del runner
import gc, torch
gc.collect(); torch.cuda.empty_cache()

runner = LlamaRunner(condition="soft",
                     few_shot_examples=FEW_SHOT_EXAMPLES)

for r in sample:
    t0 = time.time()
    pred = runner.generate(r['dialogue'])
    dt = time.time() - t0
    print(f"\n=== {r['encounter_id']}  ({dt:.1f}s) ===")
    print("--- GOLD ---")
    print(json.dumps(r['label'], indent=2))
    print("--- PRED ---")
    print(json.dumps(pred, indent=2))

Loading meta-llama/Llama-3.2-3B-Instruct for condition=soft...


Loading weights:   0%|          | 0/254 [00:00<?, ?it/s]


=== D2N002  (21.5s) ===
--- GOLD ---
{
  "name": "Andrew",
  "age": 62,
  "symptoms": [
    "joint pain",
    "knee pain"
  ],
  "duration": "over the weekend",
  "negated_symptoms": [
    "fever",
    "chills",
    "muscle aches",
    "nausea",
    "vomiting",
    "diarrhea",
    "headache",
    "fatigue",
    "weight gain"
  ],
  "history": [
    "kidney transplant",
    "hypothyroidism",
    "arthritis",
    "polycystic kidneys"
  ],
  "diagnosis": [
    "acute exacerbation of arthritis",
    "hypothyroidism"
  ],
  "treatment": [
    {
      "type": "medication",
      "detail": "ultram 50 mg every 6 hours as needed"
    },
    {
      "type": "medication",
      "detail": "continue synthroid"
    },
    {
      "type": "test",
      "detail": "order autoimmune panel"
    },
    {
      "type": "test",
      "detail": "order thyroid panel"
    },
    {
      "type": "counseling",
      "detail": "take it easy for now"
    },
    {
      "type": "follow_up",
      "detail": "talk a

In [10]:
del runner
import gc, torch
gc.collect(); torch.cuda.empty_cache()

runner = LlamaRunner(condition="medium",
                     few_shot_examples=FEW_SHOT_EXAMPLES)

for r in sample:
    t0 = time.time()
    pred = runner.generate(r['dialogue'])
    dt = time.time() - t0
    print(f"\n=== {r['encounter_id']}  ({dt:.1f}s) ===")
    print("--- GOLD ---")
    print(json.dumps(r['label'], indent=2))
    print("--- PRED ---")
    print(json.dumps(pred, indent=2))

Loading meta-llama/Llama-3.2-3B-Instruct for condition=medium...


Loading weights:   0%|          | 0/254 [00:00<?, ?it/s]


=== D2N002  (25.9s) ===
--- GOLD ---
{
  "name": "Andrew",
  "age": 62,
  "symptoms": [
    "joint pain",
    "knee pain"
  ],
  "duration": "over the weekend",
  "negated_symptoms": [
    "fever",
    "chills",
    "muscle aches",
    "nausea",
    "vomiting",
    "diarrhea",
    "headache",
    "fatigue",
    "weight gain"
  ],
  "history": [
    "kidney transplant",
    "hypothyroidism",
    "arthritis",
    "polycystic kidneys"
  ],
  "diagnosis": [
    "acute exacerbation of arthritis",
    "hypothyroidism"
  ],
  "treatment": [
    {
      "type": "medication",
      "detail": "ultram 50 mg every 6 hours as needed"
    },
    {
      "type": "medication",
      "detail": "continue synthroid"
    },
    {
      "type": "test",
      "detail": "order autoimmune panel"
    },
    {
      "type": "test",
      "detail": "order thyroid panel"
    },
    {
      "type": "counseling",
      "detail": "take it easy for now"
    },
    {
      "type": "follow_up",
      "detail": "talk a

In [11]:
del runner
gc.collect(); torch.cuda.empty_cache()

from llama_runner import load_icd10_vocab
diagnosis_vocab = load_icd10_vocab(str(icd_path))
print(f"Loaded {len(diagnosis_vocab)} ICD-10 descriptions.")

runner = LlamaRunner(condition="hard",
                     few_shot_examples=FEW_SHOT_EXAMPLES,
                     diagnosis_vocab=diagnosis_vocab)

for r in sample:
    t0 = time.time()
    pred = runner.generate(r['dialogue'])
    dt = time.time() - t0
    print(f"\n=== {r['encounter_id']}  ({dt:.1f}s) ===")
    print("--- GOLD ---")
    print(json.dumps(r['label'], indent=2))
    print("--- PRED ---")
    print(json.dumps(pred, indent=2))

Loaded 2763 ICD-10 descriptions.
Loading meta-llama/Llama-3.2-3B-Instruct for condition=hard...


Loading weights:   0%|          | 0/254 [00:00<?, ?it/s]

Hard condition: lm-format-enforcer constraining diagnosis/history to 2763 ICD-10 descriptions.

=== D2N002  (18.7s) ===
--- GOLD ---
{
  "name": "Andrew",
  "age": 62,
  "symptoms": [
    "joint pain",
    "knee pain"
  ],
  "duration": "over the weekend",
  "negated_symptoms": [
    "fever",
    "chills",
    "muscle aches",
    "nausea",
    "vomiting",
    "diarrhea",
    "headache",
    "fatigue",
    "weight gain"
  ],
  "history": [
    "kidney transplant",
    "hypothyroidism",
    "arthritis",
    "polycystic kidneys"
  ],
  "diagnosis": [
    "acute exacerbation of arthritis",
    "hypothyroidism"
  ],
  "treatment": [
    {
      "type": "medication",
      "detail": "ultram 50 mg every 6 hours as needed"
    },
    {
      "type": "medication",
      "detail": "continue synthroid"
    },
    {
      "type": "test",
      "detail": "order autoimmune panel"
    },
    {
      "type": "test",
      "detail": "order thyroid panel"
    },
    {
      "type": "counseling",
      "

In [ ]:
del runner
gc.collect(); torch.cuda.empty_cache()
print("Smoke test complete. Eyeball the outputs above. "
      "If they look reasonable, proceed to the full inference run.")

In [12]:
import os, gc, torch
os.chdir(f'{PROJECT_ROOT}/inference')

try:
    del runner
except NameError:
    pass
gc.collect(); torch.cuda.empty_cache()

!PYTHONPATH={PROJECT_ROOT}/labeler python run_inference.py \
    --labels {PROJECT_ROOT}/labeler/labels_clean.jsonl \
    --output {PROJECT_ROOT}/predictions_Alisa/ \
    --icd10-csv {PROJECT_ROOT}/inference/icd10_filtered.csv \
    --conditions baseline soft

Loaded 67 ground-truth records.
Excluded 2 few-shot examples (['D2N001', 'D2N003']); evaluating on 65 records.
Using 2 few-shot examples in soft/medium/hard prompts (baseline ignores them).

Condition: baseline
Resuming: 0 already predicted.
Loading meta-llama/Llama-3.2-3B-Instruct for condition=baseline...
`torch_dtype` is deprecated! Use `dtype` instead!
Loading weights: 100% 254/254 [00:29<00:00,  8.71it/s, Materializing param=model.norm.weight]
The following generation flags are not valid and may be ignored: ['temperature', 'top_p']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
[DEBUG RAW OUTPUT]:
```json
{
  "name": "Andrew",
  "age": 62,
  "symptoms": ["joint pain", "knee pain"],
  "duration": "over the weekend",
  "negated_symptoms": ["fever", "chills", "nausea", "vomiting", "diarrhea"],
  "history": [
    "past medical history of kidney transplant, hypothyroidism, and arthritis",
    "previous knee problems"
  ],
  "diagnosis": [
    "acute exacerbation of arthritis",
  

In [9]:
# !PYTHONPATH={PROJECT_ROOT}/labeler python run_inference.py \
#     --labels {PROJECT_ROOT}/labeler/labels_clean.jsonl \
#     --output {PROJECT_ROOT}/predictions_Alisa/ \
#     --icd10-csv {PROJECT_ROOT}/inference/icd10_filtered.csv \
#     --conditions medium hard

import os, gc, torch
os.chdir(f'{PROJECT_ROOT}/inference')

try:
    del runner
except NameError:
    pass
gc.collect(); torch.cuda.empty_cache()

!PYTHONPATH={PROJECT_ROOT}/labeler python run_inference.py \
    --labels {PROJECT_ROOT}/labeler/labels_clean.jsonl \
    --output {PROJECT_ROOT}/predictions_Alisa/ \
    --icd10-csv {PROJECT_ROOT}/inference/icd10_filtered.csv \
    --conditions medium hard


Loaded 67 ground-truth records.
Excluded 2 few-shot examples (['D2N001', 'D2N003']); evaluating on 65 records.
Diagnosis vocab (from /content/drive/MyDrive/HouseLLM/inference/icd10_filtered.csv): 2763 ICD-10 descriptions.
Using 2 few-shot examples in soft/medium/hard prompts (baseline ignores them).

Condition: medium
Resuming: 0 already predicted.
Loading meta-llama/Llama-3.2-3B-Instruct for condition=medium...
config.json: 100% 878/878 [00:00<00:00, 3.52MB/s]
tokenizer_config.json: 54.5kB [00:00, 18.3MB/s]
tokenizer.json: 9.09MB [00:00, 16.3MB/s]
special_tokens_map.json: 100% 296/296 [00:00<00:00, 1.62MB/s]
`torch_dtype` is deprecated! Use `dtype` instead!
model.safetensors.index.json: 20.9kB [00:00, 12.8MB/s]
Fetching 2 files: 100% 2/2 [00:19<00:00,  9.91s/it]
Download complete: 100% 6.43G/6.43G [00:19<00:00, 323MB/s]
Loading weights: 100% 254/254 [00:01<00:00, 129.55it/s, Materializing param=model.norm.weight]
generation_config.json: 100% 189/189 [00:00<00:00, 1.00MB/s]
The followi

In [11]:
os.chdir(f'{PROJECT_ROOT}/evaluation')

!python evaluator.py \
    --labels {PROJECT_ROOT}/labeler/labels_clean.jsonl \
    --predictions-dir {PROJECT_ROOT}/predictions_Alisa/ \
    --output {PROJECT_ROOT}/results_deterministic.json \
    --skip-judge

Loaded 67 ground-truth records.

Evaluating condition: baseline

Evaluating condition: soft

Evaluating condition: medium

Evaluating condition: hard

Results written to /content/drive/MyDrive/HouseLLM/results_deterministic.json

=== HEADLINE METRICS ===
condition      validity     sym_f1     neg_f1      tx_f1     halluc
baseline          0.031      0.267      0.333      0.000      0.085
soft              0.815      0.287      0.191      0.129      0.098
medium            1.000      0.294      0.191      0.128      0.098
hard              1.000      0.158      0.122      0.017      0.049


In [13]:
!python evaluator.py \
    --labels {PROJECT_ROOT}/labeler/labels_clean.jsonl \
    --predictions-dir {PROJECT_ROOT}/predictions_Alisa/ \
    --output {PROJECT_ROOT}/results_full.json

Loaded 67 ground-truth records.

Evaluating condition: baseline

Evaluating condition: soft
Judge failed on D2N051: Error code: 429 - {'error': {'message': 'Rate limit reached for gpt-4o in organization org-r0F7sAEIFTI2fyeezymJ0T7R on tokens per min (TPM): Limit 30000, Used 27604, Requested 2407. Please try again in 22ms. Visit https://platform.openai.com/account/rate-limits to learn more.', 'type': 'tokens', 'param': None, 'code': 'rate_limit_exceeded'}}

Evaluating condition: medium
Judge failed on D2N008: Error code: 429 - {'error': {'message': 'Rate limit reached for gpt-4o in organization org-r0F7sAEIFTI2fyeezymJ0T7R on tokens per min (TPM): Limit 30000, Used 29123, Requested 2597. Please try again in 3.44s. Visit https://platform.openai.com/account/rate-limits to learn more.', 'type': 'tokens', 'param': None, 'code': 'rate_limit_exceeded'}}

Evaluating condition: hard

Results written to /content/drive/MyDrive/HouseLLM/results_full.json

=== HEADLINE METRICS ===
condition      va

In [15]:
import json

with open(f'{PROJECT_ROOT}/evaluation/results_full.json') as f:
    results = json.load(f)

# Print headline comparison table
print(f"{'condition':<12} {'validity':>10} {'sym_f1':>10} {'tx_f1':>10} {'halluc':>10}")
for cond, r in results.items():
    det = r['deterministic']
    print(f"{cond:<12} {r['validity_rate']:>10.3f} "
          f"{det.get('symptoms_f1', 0):>10.3f} "
          f"{det.get('treatment_f1', 0):>10.3f} "
          f"{det.get('hallucination_rate', 0):>10.3f}")

condition      validity     sym_f1      tx_f1     halluc
baseline          0.031      0.267      0.000      0.085
soft              0.815      0.287      0.129      0.098
medium            1.000      0.294      0.128      0.098
hard              1.000      0.158      0.017      0.049


In [16]:
# Generate plots from results_full.json
import os
os.chdir(f'{PROJECT_ROOT}/evaluation')

!python make_plots.py \
    --results {PROJECT_ROOT}/evaluation/results_full.json \
    --out {PROJECT_ROOT}/evaluation/plots_Alisa/

Generating plots in /content/drive/MyDrive/HouseLLM/evaluation/plots_Alisa/ ...

  ✓ 01_headline_overview.png
  ✓ 02_per_field_f1.png
  ✓ 03_diagnosis_distribution.png
  ✓ 04_error_decomposition.png
  ✓ 05_llm_judge_heatmap.png
  ✓ 06_latency_vs_f1.png

=== Paired Wilcoxon tests (adjacent conditions) ===
comparing per-record values across the 65 records

metric                baseline→soft      soft→medium    medium→hard      
symptoms_f1           p=0.925 ns   p=0.713 ns   p=0.000 ***  
/usr/local/lib/python3.12/dist-packages/scipy/stats/_wilcoxon.py:178: RuntimeWarning: invalid value encountered in scalar divide
  z = (r_plus - mn) / se
negated_symptoms_f1   p=0.099 ns   p=nan ns     p=0.106 ns   
history_f1            p=0.000 ***  p=0.715 ns   p=0.000 ***  
diagnosis_f1          p=0.000 ***  p=0.180 ns   p=0.000 ***  
treatment_f1          p=0.000 ***  p=0.917 ns   p=0.000 ***  
hallucination_rate    p=0.404 ns   p=nan ns     p=0.073 ns   

Done. 6 plots in /content/drive/MyDrive/Ho